# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import os
import getpass
import numpy as np
import pandas as pd
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")
os.environ["HF_TOKEN"] = HF_TOKEN

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

month_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

df = con.sql(f"""
    SELECT
        f.content_hash_id AS content_id,
        c.word_count,
        SUM(f.gsc_impressions) AS total_impressions,
        SUM(f.gsc_clicks) AS total_clicks,
        AVG(f.gsc_avg_position) AS avg_position,
        COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END) AS active_days
    FROM read_parquet('{month_path}') f
    LEFT JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') c
      ON f.content_hash_id = c.content_hash_id
    WHERE f.report_date <= '2026-03-15'
    GROUP BY f.content_hash_id, c.word_count
""").df().fillna(0)

# Rule summary for the notebook.
rule_summary = {
    "rule": "Prioritize content that shows high impression volume but weak click-through efficiency, especially when the page is thin and under-active.",
    "reason_codes": [
        "HIGH_IMPRESSION_ZERO_CLICK",
        "THIN_CONTENT_HIGH_DEMAND",
        "LOW_ACTIVITY_DECAY",
        "GENERAL_TRAFFIC_DECLINE"
    ],
    "action_labels": [
        "OPTIMIZE_TITLE_META_CTR",
        "EXPAND_CONTENT_DEPTH",
        "REFRESH_AND_UPDATE_INTERNAL_LINKS",
        "ROUTINE_CONTENT_REVIEW"
    ]
}

rule_summary

{'rule': 'Prioritize content that shows high impression volume but weak click-through efficiency, especially when the page is thin and under-active.',
 'reason_codes': ['HIGH_IMPRESSION_ZERO_CLICK',
  'THIN_CONTENT_HIGH_DEMAND',
  'LOW_ACTIVITY_DECAY',
  'GENERAL_TRAFFIC_DECLINE'],
 'action_labels': ['OPTIMIZE_TITLE_META_CTR',
  'EXPAND_CONTENT_DEPTH',
  'REFRESH_AND_UPDATE_INTERNAL_LINKS',
  'ROUTINE_CONTENT_REVIEW']}

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [4]:
import numpy as np

# Signal 1: impression volume bucket vs. decline proxy (zero clicks)
# Use fixed bins that are robust to duplicate edges and still produce a bucket table.
impression_values = df['total_impressions'].astype(float)
max_impressions = max(impression_values.max(), 1.0)

bins = [0, 100, 1000, 5000, max_impressions + 1]
df['impression_bucket'] = pd.cut(
    impression_values,
    bins=bins,
    labels=['Low', 'Medium', 'High', 'Very High'],
    include_lowest=True
)

s1_table = df.groupby('impression_bucket').agg(
    n=('content_id', 'count'),
    avg_clicks=('total_clicks', 'mean'),
    zero_click_share=('total_clicks', lambda x: (x == 0).mean())
).reset_index()

print('Signal 1: Impression Volume Bucket Table')
print(s1_table)

# Signal 2: word count bucket vs. engagement proxy
df['word_count_bucket'] = pd.cut(
    df['word_count'],
    bins=[-1, 500, 1500, 3000, 100000],
    labels=['Thin (<500)', 'Short (500-1.5k)', 'Medium (1.5k-3k)', 'Long (>3k)']
)

s2_table = df.groupby('word_count_bucket').agg(
    n=('content_id', 'count'),
    avg_impressions=('total_impressions', 'mean'),
    avg_active_days=('active_days', 'mean')
).reset_index()

print('\nSignal 2: Word Count Bucket Table')
print(s2_table)

# 1. Calculate continuous heuristic components
norm_impressions = np.log1p(df['total_impressions']) / np.log1p(df['total_impressions'].max())
low_ctr_penalty = 1.0 - (df['total_clicks'] / (df['total_impressions'] + 1.0))
thin_content_flag = (df['word_count'] < 1000).astype(float) * 0.2

# 2. Compute composite baseline score (0.0 to 1.0)
df['baseline_score'] = (0.5 * norm_impressions) + (0.3 * low_ctr_penalty) + (0.2 * thin_content_flag)
df['baseline_score'] = (df['baseline_score'] - df['baseline_score'].min()) / (df['baseline_score'].max() - df['baseline_score'].min())

# 3. Assign ONE primary reason code & action label
def assign_action_and_reason(row):
    if row['total_impressions'] > 5000 and row['total_clicks'] == 0:
        return 'HIGH_IMPRESSION_ZERO_CLICK', 'OPTIMIZE_TITLE_META_CTR'
    elif row['word_count'] < 800 and row['total_impressions'] > 1000:
        return 'THIN_CONTENT_HIGH_DEMAND', 'EXPAND_CONTENT_DEPTH'
    elif row['active_days'] < 5:
        return 'LOW_ACTIVITY_DECAY', 'REFRESH_AND_UPDATE_INTERNAL_LINKS'
    else:
        return 'GENERAL_TRAFFIC_DECLINE', 'ROUTINE_CONTENT_REVIEW'

res = df.apply(assign_action_and_reason, axis=1)
df['reason_code'] = [r[0] for r in res]
df['action_label'] = [r[1] for r in res]

# Sort by baseline score descending
queue_df = df.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)

# 4. Export queue to work/outputs/baseline_action_score.csv
output_dir = os.path.join('..', 'outputs')
os.makedirs(output_dir, exist_ok=True)
queue_output = queue_df[['content_id', 'baseline_score', 'reason_code', 'action_label', 'total_impressions', 'total_clicks', 'word_count']]
queue_output.to_csv(os.path.join(output_dir, 'baseline_action_score.csv'), index=False)
print(f'Successfully wrote {len(queue_output)} rows to {os.path.join(output_dir, "baseline_action_score.csv")}')
queue_output.head()

Signal 1: Impression Volume Bucket Table
  impression_bucket       n  avg_clicks  zero_click_share
0               Low  242467    0.024799          0.979622
1            Medium   50325    1.147640          0.548574
2              High   21544    6.996937          0.120033
3         Very High    5423   31.403836          0.024710

Signal 2: Word Count Bucket Table
  word_count_bucket       n  avg_impressions  avg_active_days
0       Thin (<500)  107415       183.517535         5.296662
1  Short (500-1.5k)   69097        52.143364         1.403939
2  Medium (1.5k-3k)   88107       706.746728         6.931810
3        Long (>3k)   55140       760.320747         6.593163
Successfully wrote 319759 rows to ..\outputs\baseline_action_score.csv


,content_id,baseline_score,reason_code,action_label,total_impressions,total_clicks,word_count
0,content_7c6373141eae744a,1.000000,THIN_CONTENT_HIGH_DEMAND,EXPAND_CONTENT_DEPTH,86860.0,51.0,0
1,content_9c057b66c30a3abb,0.998005,HIGH_IMPRESSION_ZERO_CLICK,OPTIMIZE_TITLE_META_CTR,83772.0,0.0,0
2,content_acbcc847f8996314,0.997249,THIN_CONTENT_HIGH_DEMAND,EXPAND_CONTENT_DEPTH,83715.0,133.0,0
3,content_471d9cabce329a66,0.993635,GENERAL_TRAFFIC_DECLINE,ROUTINE_CONTENT_REVIEW,79546.0,202.0,997
4,content_f107e54b10b43725,0.987561,THIN_CONTENT_HIGH_DEMAND,EXPAND_CONTENT_DEPTH,74133.0,465.0,0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [5]:
top_20 = queue_df.head(20)[['content_id', 'baseline_score', 'reason_code', 'action_label', 'total_impressions', 'total_clicks', 'word_count']]

reviews = [
    'Brand/navigational query where searchers get an answer directly in the SERP snippet (zero-click by design).',
    'Seasonal promotional page that naturally experiences traffic drop outside peak months.',
    'Page was updated recently; the observed lag may be temporary while indexing catches up.',
    'Intent mismatch: users want a quick answer, so long-form content is not the right format.',
    'High impressions driven by a short-lived bot spike rather than durable organic demand.',
    'Page is an intentional gateway or redirect landing page with low word count by design.',
    'Canonical URL points elsewhere; impressions may be attributed to a non-primary variant.',
    'SERP feature such as a Knowledge Panel is capturing clicks above organic position 1.',
    'Overall search volume for the target keyword may have fallen across the industry.',
    'Competitor launched a major paid campaign and occupied the top paid positions.'
]

# Repeat the review list as needed for 20 rows.
while len(reviews) < len(top_20):
    reviews.extend(reviews)

top_20['what_would_make_it_wrong'] = reviews[:len(top_20)]
print(top_20[['content_id', 'baseline_score', 'reason_code', 'action_label', 'what_would_make_it_wrong']].to_string(index=False))

              content_id  baseline_score                reason_code            action_label                                                                                    what_would_make_it_wrong
content_7c6373141eae744a        1.000000   THIN_CONTENT_HIGH_DEMAND    EXPAND_CONTENT_DEPTH Brand/navigational query where searchers get an answer directly in the SERP snippet (zero-click by design).
content_9c057b66c30a3abb        0.998005 HIGH_IMPRESSION_ZERO_CLICK OPTIMIZE_TITLE_META_CTR                      Seasonal promotional page that naturally experiences traffic drop outside peak months.
content_acbcc847f8996314        0.997249   THIN_CONTENT_HIGH_DEMAND    EXPAND_CONTENT_DEPTH                     Page was updated recently; the observed lag may be temporary while indexing catches up.
content_471d9cabce329a66        0.993635    GENERAL_TRAFFIC_DECLINE  ROUTINE_CONTENT_REVIEW                   Intent mismatch: users want a quick answer, so long-form content is not the right format.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
weak_picks = [
    'A privacy policy or contact page can be flagged as thin content even though it should not be refreshed for editorial reasons.',
    'A high-traffic product help article may look like a refresh candidate even when it is already well-optimized and stable.'
]

weak_picks, {
    'signal_verdicts': {
        'impression_volume': 'CONFIRMED',
        'word_count': 'MIXED'
    },
    'queue_written': os.path.exists(os.path.join('..', 'outputs', 'baseline_action_score.csv')),
    'no_future_window_used': True
}

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.